
# 10 — TCN, Transformer, DLinear and N-BEATS

Notebook 08 left a specific question open. It found that **lag depth mattered more than
architecture**: four lags to twenty-four bought 18% RMSE on GP for any model, while the
best architecture bought 9% on top of that, on one operator only.

If *which* past steps matter is the live question, the architectures that address it
directly had not been tried. Each model here is included because it tests a particular
reading of that finding — not because it is fashionable.

| model | the reading it tests | source |
|---|---|---|
| **DLinear / NLinear** | that almost none of this needs depth at all — a one-layer linear model on a decomposed series was enough to beat every Transformer the authors tested | Zeng et al., AAAI 2023 (arXiv:2205.13504) |
| **TCN** | that the win belongs to *convolution over a deep window*. The original's "CNN" is a single undilated `Conv1D`; dilated causal convolutions are the principled version of the architecture that already won on GP | Bai, Kolter & Koltun, arXiv:1803.01271 |
| **Transformer** | that the model should *choose* which lags matter instead of weighting all 24 alike. The attention map is itself a result, checkable against the measured daily period | Vaswani et al., NeurIPS 2017 |
| **N-BEATS / NBEATSx** | that a pure forecasting architecture with a learned basis beats feature engineering. NBEATSx adds the exogenous block; plain N-BEATS is kept as the **univariate control** | Oreshkin et al., ICLR 2020; Olivares et al., IJF 2023 |

## Stated before the results, so it cannot be fitted afterwards

On ~900 points across 55 days, **the small models should win.** Elsayed et al.
(arXiv:2101.02118) found gradient boosting on a windowed representation matching
state-of-the-art deep models on benchmarks two orders of magnitude larger than this
trace. N-BEATS and the Transformer are here because *a negative result at this sample
size is a result* — it is evidence for the lag-depth reading, not against the
architectures.

## What is deliberately absent, and why

- **PatchTST** — patching a 24-step window yields about three tokens. The mechanism the
  paper depends on cannot operate at our lookback.
- **Informer, Autoformer** — built for horizons of 96–720. Ours is 1–17.
- **The full Temporal Fusion Transformer** — needs many related series. We borrow its
  covariate taxonomy, not its architecture.

These are recorded so the omissions read as decisions rather than oversights. All of it
is in `docs/PROVENANCE.md` with verified citations.

In [ ]:
# --- Bootstrap: works locally, on Colab and on Kaggle ----------------------
# RUN THIS CELL FIRST, and re-run it after any kernel restart. Every later cell
# depends on it. If it fails, the next cell fails with "No module named bwalloc",
# which looks like a different problem but is not.
#
# On a hosted runtime it clones the repo (or pulls, on a re-run) and installs the
# package into the session, so `import bwalloc` keeps working even from a cell you
# run on its own after restarting. The repository is public; no token is needed.
import os, subprocess, sys, warnings
from pathlib import Path

warnings.filterwarnings("ignore")
REPO = "https://github.com/sad-code-at/bwalloc.git"

def _find_root(start: Path):
    node = start
    while not (node / "src" / "bwalloc").exists() and node != node.parent:
        node = node.parent
    return node if (node / "src" / "bwalloc").exists() else None

ROOT = _find_root(Path.cwd())

if ROOT is None:
    # A repo uploaded as a Kaggle Dataset mounts read-only here; prefer it if present.
    for candidate in Path("/kaggle/input").glob("*/src/bwalloc"):
        ROOT = candidate.parent.parent
        break

if ROOT is None:
    on_kaggle = Path("/kaggle/working").exists()
    target = Path("/kaggle/working/bwalloc") if on_kaggle else Path("/content/bwalloc")
    if _find_root(target) is None:
        if os.system("git clone -q " + REPO + " " + str(target)) != 0 or _find_root(target) is None:
            raise RuntimeError(
                "Clone failed. On Kaggle, switch Internet ON in the right sidebar "
                "(Settings > Internet), then re-run this cell. See docs/KAGGLE.md."
            )
        print("cloned", REPO)
    else:
        os.system("git -C " + str(target) + " pull -q --ff-only")
        print("pulled latest into", target)
    os.chdir(target)
    ROOT = target
    # Install into the session so `import bwalloc` survives a kernel restart and
    # does not depend on this cell having set sys.path. --no-deps deliberately:
    # Kaggle and Colab curate their own numpy/pandas and we must not disturb them.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"],
                   cwd=str(ROOT), capture_output=True)

sys.path.insert(0, str(ROOT / "src"))

try:
    import xgboost  # noqa: F401
except ImportError:
    !pip install -q xgboost

import numpy as np, pandas as pd, matplotlib.pyplot as plt
import bwalloc as bw
from bwalloc.plots import use_paper_style

bw.set_seed()
use_paper_style()
pd.set_option("display.width", 200)
RESULTS = ROOT / "experiments" / "results"
# Scratch output for the exploratory notebooks. Only 07_paper_figures writes into
# paper/figures -- otherwise running notebook 00 or 05 silently overwrites a figure
# the paper cites, which is exactly the kind of drift this project exists to remove.
FIGURES = ROOT / "notebooks" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)
_data = sorted((ROOT / "data").glob("*.csv"))
_res = sorted(RESULTS.glob("*.csv"))
print("bwalloc", bw.__version__, "at", ROOT)
print(f"  data/    {len(_data)} csv  ({', '.join(f.name for f in _data) or 'MISSING'})")
print(f"  results/ {len(_res)} csv")
if not _data:
    raise RuntimeError(
        "The trace CSVs are missing, so nothing will run. Re-run this cell to "
        "re-clone, or check that the repository was fetched completely."
    )


## Setup

Every model reads the same design matrix: full corrected features, demand lags 1–24, and
each context flag and daily harmonic at those same 24 lags. Three of these architectures
are univariate in their source papers; implemented literally they would have read channel
0 and ignored the rest, scoring plausibly the whole time. Covariates therefore enter
structurally, and `NBeats` is retained — labelled — as the univariate control.

In [ ]:

from bwalloc.architectures import (MODERN_ARCHITECTURES, TransformerForecaster,
                                   channel_permutation_importance, modern_models)
from bwalloc.baselines import SeasonalNaive, standard_baselines
from bwalloc.data import load, sampling_profile
from bwalloc.evaluate import beats_baseline, dm_matrix, run_backtest, summarise
from bwalloc.features import assert_no_leakage, build_features
from bwalloc.models import default_point_models
from bwalloc.sequence import channel_window, covariate_sequence_models, full_feature_config
from bwalloc.splits import rolling_origin

OPERATOR = "gp"          # switch to "robi" and re-run
N_FOLDS, LOOKBACK, EPOCHS = 8, 24, 60

df = load(OPERATOR)
profile = sampling_profile(df)
config = full_feature_config(LOOKBACK, covariates=True)
assert_no_leakage(df, profile, config)

X, y = build_features(df, profile, config)
spec = channel_window(X, LOOKBACK)
folds = rolling_origin(len(y), n_folds=N_FOLDS)
print(f"{OPERATOR.upper()}: {len(X)} rows, {spec.n_channels} channels x {LOOKBACK} lags "
      f"+ {len(spec.static)} static, {N_FOLDS} folds")
print("architectures:", ", ".join(MODERN_ARCHITECTURES))


## Size first

Parameter count against training rows is the number to have in mind before reading any
accuracy. Each fold fits on roughly 350–800 rows.

In [ ]:

sizes = []
head = X.iloc[:120]
for m in modern_models(lookback=LOOKBACK, epochs=1):
    m.fit(head, y.iloc[:120])
    sizes.append({"model": m.name, "parameters": m.n_parameters})
sizes = pd.DataFrame(sizes).sort_values("parameters")
sizes["params_per_training_row"] = sizes["parameters"] / len(folds[0].train)
sizes.round(1)

## The comparison

In [ ]:

import time

models = list(default_point_models())
models += covariate_sequence_models(lookback=LOOKBACK, epochs=30)
models += modern_models(lookback=LOOKBACK, epochs=EPOCHS)

baselines = standard_baselines(y.to_numpy(), profile.daily_period)
baselines.append(SeasonalNaive(y.to_numpy(), period=24))

started = time.time()
per_fold, predictions = run_backtest(
    X, y, folds, models=models, baselines=baselines, season_lag=profile.daily_period)
summary = beats_baseline(summarise(per_fold))
print(f"{time.time() - started:.0f}s")
summary[["model", "rmse_mean", "rmse_std", "mae_mean", "mase_mean",
         "vs_persistence", "beats_persistence"]].round(3)


## Which differences survive a significance test?

Ten-plus models means dozens of pairwise comparisons, and at α = 0.05 uncorrected two or
three spurious wins are expected by construction. Benjamini–Hochberg controls that.

In [ ]:

dm = dm_matrix(predictions, horizon=1)
leader = summary["model"].iloc[0]
pairs = dm[(dm["model_a"] == leader) | (dm["model_b"] == leader)]
print(f"leader: {leader};  {int(pairs['significant_fdr'].sum())} of {len(pairs)} "
      f"comparisons against it survive BH correction")
pairs[["model_a", "model_b", "dm_stat", "p_value", "significant_fdr", "winner"]]


## Is it really reading more than the lags?

The direct measurement rather than the assumption. Each channel is shuffled across the
window — all 24 lags of it together, because shuffling one lag of one flag while leaving
its neighbours intact would leave the information almost entirely recoverable — and the
RMSE cost recorded. A channel that costs nothing is a channel the model ignored.

The audit found most context flags inert on the *level* of demand, so zeros here are a
live possibility and worth measuring rather than assuming away.

In [ ]:

last = folds[-1]
fit_idx = np.concatenate([last.train, last.calib])
X_fit, y_fit = X.iloc[fit_idx], y.iloc[fit_idx]
X_test, y_test = X.iloc[last.test], y.iloc[last.test]

rows = []
for m in modern_models(lookback=LOOKBACK, epochs=EPOCHS):
    if m.name not in ("tcn", "transformer", "nbeatsx"):
        continue
    m.fit(X_fit, y_fit)
    imp = channel_permutation_importance(m, X_test, y_test, spec, n_repeats=5)
    imp.insert(0, "model", m.name)
    rows.append(imp)

importance = pd.concat(rows, ignore_index=True)
wide = importance.pivot(index="channel", columns="model", values="importance")
wide.sort_values("tcn", ascending=False).round(3)

In [ ]:

top = wide.mean(axis=1).sort_values().tail(10)
fig, ax = plt.subplots(figsize=(7.5, 4.4))
colours = ["#c0392b" if c == "demand" else "#2f6f9f" for c in top.index]
ax.barh(top.index, top.values, color=colours)
ax.set_xlabel("RMSE cost of shuffling this channel")
ax.set_title(f"{OPERATOR.upper()} — what each input channel is worth (mean over models)")
ax.axvline(0, color="#444", lw=0.8)
fig.tight_layout()
fig.savefig(FIGURES / f"fig14_channel_importance_{OPERATOR}.png", dpi=200,
            bbox_inches="tight")


## Which lags does attention choose?

This is the figure that speaks directly to notebook 08's finding. If lag depth mattered
because *particular* distant lags carry signal, attention should concentrate there — and
the obvious candidate is the measured daily period, 17 samples on GP and 15 on Robi. If
instead attention spreads evenly, the depth was buying general smoothing rather than
seasonality, which is a different story.

In [ ]:

tr = TransformerForecaster(lookback=LOOKBACK, epochs=EPOCHS).fit(X_fit, y_fit)
tr.predict(X_test)
weights = tr.attention_by_lag          # oldest first
lags = np.arange(LOOKBACK, 0, -1)

fig, ax = plt.subplots(figsize=(7.5, 3.6))
ax.bar(lags, weights, color="#2f6f9f")
ax.axvline(profile.daily_period, color="#c0392b", ls="--", lw=1.4,
           label=f"measured daily period = {profile.daily_period} samples")
ax.axhline(1 / LOOKBACK, color="#666", ls=":", lw=1.2, label="uniform attention")
ax.set_xlabel("lag (samples back)")
ax.set_ylabel("mean attention")
ax.set_title(f"{OPERATOR.upper()} — where the most recent position attends")
ax.invert_xaxis()
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES / f"fig13_attention_{OPERATOR}.png", dpi=200, bbox_inches="tight")

## Does complexity buy accuracy?

In [ ]:

merged = sizes.merge(summary[["model", "rmse_mean"]], on="model")
fig, ax = plt.subplots(figsize=(7, 4.4))
ax.scatter(merged["parameters"], merged["rmse_mean"], s=70, color="#2f6f9f", zorder=3)
for _, r in merged.iterrows():
    ax.annotate(r["model"], (r["parameters"], r["rmse_mean"]),
                textcoords="offset points", xytext=(6, 4), fontsize=9)
ax.axhline(summary.loc[summary["model"] == "persistence", "rmse_mean"].iloc[0],
           color="#c0392b", ls="--", lw=1.3, label="persistence")
ax.axhline(summary.loc[summary["model"] == "random_forest", "rmse_mean"].iloc[0],
           color="#666", ls=":", lw=1.3, label="random forest")
ax.set_xscale("log")
ax.set_xlabel("parameters (log scale)")
ax.set_ylabel("RMSE")
ax.set_title(f"{OPERATOR.upper()} — parameter count against accuracy")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES / "fig15_params_vs_rmse.png", dpi=200, bbox_inches="tight")

## Figure: the field

In [ ]:

field = summary[~summary["model"].isin(["train_mean"])].sort_values("rmse_mean")
new = set(MODERN_ARCHITECTURES)
colour = ["#2f6f9f" if m in new else "#9aa5b1" for m in field["model"]]

fig, ax = plt.subplots(figsize=(8, 5.2))
ax.barh(field["model"], field["rmse_mean"], xerr=field["rmse_std"], capsize=3,
        color=colour)
ax.axvline(summary.loc[summary["model"] == "persistence", "rmse_mean"].iloc[0],
           color="#c0392b", ls="--", lw=1.3, label="persistence")
ax.invert_yaxis()
ax.set_xlabel("RMSE (mean +/- sd over 8 folds)")
ax.set_title(f"{OPERATOR.upper()} — modern architectures (blue) against the incumbents")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES / f"fig12_architectures_{OPERATOR}.png", dpi=200, bbox_inches="tight")


## Both operators

`experiments/run_architectures.py` runs all of this for GP and Robi.

In [ ]:

frames = []
for op in ("gp", "robi"):
    path = RESULTS / f"arch_{op}_summary.csv"
    if path.exists():
        s = pd.read_csv(path)[["model", "rmse_mean"]]
        frames.append(s.assign(operator=op))
if frames:
    both = pd.concat(frames)
    display(both.pivot(index="model", columns="operator", values="rmse_mean").round(3))
else:
    print("run experiments/run_architectures.py to fill this in")


## What to take away

Fill these in from your own run rather than from the prose — the point of the notebook is
that the numbers decide.

1. **Parameters against evidence.** Every model here fits on 350–800 rows. Read the
   parameter column before the accuracy column.
2. **The linear controls are the test of everything else.** If DLinear or NLinear sits
   inside the spread of the deep models, then at this sample size the architecture is not
   what is carrying the result — and that conclusion is worth more than a ranking.
3. **Attention says what the model chose to read.** Concentration near the measured daily
   period would corroborate notebook 08's lag-depth finding; a flat map would say the
   depth was buying smoothing instead.
4. **The permutation table is the covariate audit.** It is the measurement behind "the
   models really do read more than the lags", and a zero there is a finding, not a bug.

Notebook 11 removes the last uncontrolled variable: every number so far comes from a
model at its **defaults**.